# 07 — Optuna hyperparameter tuning

Tune ExtraTreesRegressor, CatBoostRegressor, and SVR with five-fold GroupKFold by scaffold, using training compounds only. Columns with missing or infinite values in the training partition are dropped. Within each fold, constant and highly correlated features are removed and ExtraTrees ranks features using only that fold’s training rows. The fold uses its top 250 features. Feature names may differ between folds; the feature-selection procedure and feature count stay the same. Optuna minimizes mean validation RMSE; default is 30 trials per model.

After tuning, each best-parameter model is refitted across the same five training folds and saves out-of-fold predictions and metrics. These are useful training-only diagnostics but not an independent estimate: the folds were used to choose the parameters. The locked scaffold test set remains untouched and is the final generalization check. Stage 08 selects the fixed 250-feature set using all training rows and refits the tuned models. No imputation is used. SVR scaling is fit within each training fold.

Optuna resumes saved studies. Set `optuna_trials` in `config.json` (default 30 trials per model) before the first run.

In [4]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: c:\Users\USER\Documents\E\Code


In [ ]:
from src.optuna_tuning import run
tuning_results = run()
tuning_results

Optuna: ExtraTreesRegressor, 5-fold GroupKFold, up to 30 trials


[I 2026-10-21 00:32:02,008] A new study created in RDB with name: brd4_56225827313f_ExtraTreesRegressor
[I 2026-10-21 00:35:19,980] Trial 0 finished with value: 0.6018391806759161 and parameters: {'n_estimators': 500, 'max_features': 0.9704285838459497, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_depth': 30, 'bootstrap': False}. Best is trial 0 with value: 0.6018391806759161.
[I 2026-10-21 00:42:21,354] Trial 1 finished with value: 0.5868411950704931 and parameters: {'n_estimators': 900, 'max_features': 0.899465584480253, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': False}. Best is trial 1 with value: 0.5868411950704931.
[I 2026-10-21 00:44:20,196] Trial 2 finished with value: 0.6045885139386746 and parameters: {'n_estimators': 500, 'max_features': 0.619817105976215, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_depth': 40, 'bootstrap': False}. Best is trial 1 with value: 0.5868411950704931.
[I 2026-10-21 00:47:10,999] Trial 3 finished wi

Optuna: CatBoostRegressor, 5-fold GroupKFold, up to 30 trials


[I 2026-10-21 02:02:52,799] A new study created in RDB with name: brd4_56225827313f_CatBoostRegressor
[I 2026-10-21 02:35:57,757] Trial 0 finished with value: 0.6053653414862075 and parameters: {'iterations': 500, 'depth': 10, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 2.385234757844707, 'random_strength': 0.31203728088487304}. Best is trial 0 with value: 0.6053653414862075.
[I 2026-10-21 02:36:40,692] Trial 1 finished with value: 0.6150271220964643 and parameters: {'iterations': 400, 'depth': 4, 'learning_rate': 0.13394334706750485, 'l2_leaf_reg': 2.416482602989751, 'random_strength': 1.416145155592091}. Best is trial 0 with value: 0.6053653414862075.
[I 2026-10-21 02:56:33,148] Trial 2 finished with value: 0.6200085154195396 and parameters: {'iterations': 300, 'depth': 10, 'learning_rate': 0.12106896936002161, 'l2_leaf_reg': 0.30803400529839686, 'random_strength': 0.36364993441420124}. Best is trial 0 with value: 0.6053653414862075.
[I 2026-10-21 02:58:24,046] Trial 3 finis

Optuna: SVR, 5-fold GroupKFold, up to 30 trials


[I 2026-10-21 07:27:57,085] A new study created in RDB with name: brd4_56225827313f_SVR


In [5]:
import json
from pathlib import Path
import optuna

db = PROJECT_ROOT / "outputs/07_optuna_tuning/models/optuna_studies.sqlite3"
print("Looking for database:", db)

if not db.exists():
    raise FileNotFoundError(f"Database not found: {db}")

storage = f"sqlite:///{db.resolve().as_posix()}"
study_names = optuna.study.get_all_study_names(storage=storage)

params_by_model = {}
for model in ("ExtraTreesRegressor", "CatBoostRegressor"):
    matches = [
        name for name in study_names
        if name.endswith(f"_{model}") and name.startswith("brd4_56225827313f")
    ]
    if len(matches) != 1:
        raise RuntimeError(f"Expected one saved study for {model}; found: {matches}")

    study = optuna.load_study(study_name=matches[0], storage=storage)
    params_by_model[model] = {
        "params": study.best_params,
        "best_cv_rmse": study.best_value,
        "trial_number": study.best_trial.number,
        "study_name": study.study_name,
    }

out = PROJECT_ROOT / "outputs/07_optuna_tuning/tables/optuna_best_params.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps({"params_by_model": params_by_model}, indent=2))
print("Saved:", out)

Looking for database: c:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\models\optuna_studies.sqlite3
Saved: c:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\tables\optuna_best_params.json
